# DeNovoGen
Train a PyTorch LSTM on SMILES, compare canonical and randomized training views, then fine-tune a shared prior under six REINFORCE rewards. The final cells generate molecular CSVs, check novelty against exact references, and measure throughput.

**First GPU run:** use the default `small` profile. **Portfolio experiment:** choose `medium` for three RL seeds, a complete source index, and a 100k-attempt benchmark. **Offline software check:** choose `smoke` for synthetic data and CPU-sized models. The notebook embeds all project files. Internet is needed for unavailable packages and the original dataset unless you provide them locally.

Run the cells in order. For a long run, choose a persistent **RUN_ROOT** before starting. The notebook and the CLI share the same checkpoints. Kaggle can save the writable run directory as notebook output. In Colab, a mounted Drive path can persist across sessions.

The first code cell contains the settings, project extraction, and dependency checks. Run it once per kernel session.

In [ ]:
import os, sys
from pathlib import Path

PROFILE = os.environ.get("DENOVOGEN_PROFILE", "small")   # smoke / small / medium / full
SPLIT = "random"                                    # random / scaffold
SOURCE_PATH = os.environ.get("DENOVOGEN_SOURCE", "")     # Optional .smi directory or file
DEVICE = os.environ.get("DENOVOGEN_DEVICE", "auto")     # auto / cuda / cpu
FULL_SOURCE_INDEX = None                            # None uses profile default; True scans all source rows
EXPORT_RESUMABLE_ZIP = False                        # May require several GB for large experiments

default_root = (Path("/kaggle/working") if Path("/kaggle/working").is_dir()
                else Path("/content") if Path("/content").is_dir() else Path.cwd())
WORK = Path(os.environ.get("DENOVOGEN_WORK", str(default_root / "denovogen_work"))).expanduser().resolve()
RUN_ROOT = Path(os.environ.get("DENOVOGEN_RUN_ROOT", str(WORK / "runs"))).expanduser().resolve()
# Example for a mounted Colab drive: RUN_ROOT = Path("/content/drive/MyDrive/denovogen_runs")
if PROFILE not in {"smoke","small","medium","full"} or SPLIT not in {"random","scaffold"}:
    raise ValueError("Choose a listed profile and split.")
WORK.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Run storage:", RUN_ROOT)

import base64, hashlib, io, zipfile
from pathlib import PurePosixPath
packed = base64.b64decode("...")
checksum = "075320b09586ef1898848379cb0fe175d29709ac0b4d2b69001aee763a6ddf90"
if hashlib.sha256(packed).hexdigest() != checksum:
    raise ValueError("Embedded project failed its checksum.")
PROJECT = WORK / ("source_" + checksum[:12])
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(packed)) as archive:
    for name in archive.namelist():
        relative = PurePosixPath(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Invalid embedded file path.")
        target = PROJECT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(archive.read(name))
print("Source files:", PROJECT)

import subprocess, json, shutil
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement

if sys.version_info < (3,11):
    raise RuntimeError("Use Python 3.11 or newer (3.11/3.12 for the exact audited RDKit release).")
def command(args, cwd=None):
    if cwd is None:
        cwd = PROJECT
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, args)

missing = []
for line in (PROJECT/"requirements.txt").read_text().splitlines():
    req = Requirement(line)
    if req.marker and not req.marker.evaluate():
        continue
    try:
        ok = version(req.name) in req.specifier
    except PackageNotFoundError:
        ok = False
    if not ok:
        missing.append(line)
if missing:
    command([sys.executable, "-m", "pip", "install", *missing])
command([sys.executable,"-c","import torch,rdkit; print('PyTorch',torch.__version__,'RDKit',rdkit.__version__,'GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"])
command([sys.executable,"-m","unittest","discover","-s","tests","-v"])

## Choose the corpus
The default resolves the pinned upstream commit and checks out its dataset directory. On Kaggle with internet disabled, attach the original dataset as an input and set SOURCE_PATH. The small profile scans the 242 files to draw a representative subset; it avoids a chemically biased first-shard sample.

In [ ]:
PIN = "bd1bb100871c9c19567fd05a9e5d41b010fe3010"
if SOURCE_PATH:
    source = Path(SOURCE_PATH).expanduser().resolve()
elif PROFILE == "smoke":
    source = PROJECT/"examples"/"smoke.smi"
else:
    repo = WORK / ("upstream_" + PIN[:10])
    repo.mkdir(exist_ok=True)
    def git(*args):
        command(["git","-C",repo,*args])
    if not (repo/".git").exists():
        git("init","--quiet")
    remote = subprocess.run(["git","-C",repo,"remote","get-url","origin"],capture_output=True,text=True)
    if remote.returncode:
        git("remote","add","origin","https://github.com/jitendra663735/molgen-lstm-rl.git")
    if subprocess.run(["git","-C",repo,"cat-file","-e",PIN+"^{commit}"],capture_output=True).returncode:
        git("fetch","--depth","1","origin",PIN)
    git("restore","--source",PIN,"--worktree","--","dataset")
    source = repo/"dataset"
if not source.exists():
    raise FileNotFoundError(source)
cfg = json.loads((PROJECT/"configs"/(PROFILE+".json")).read_text())
cfg["split"] = SPLIT
if FULL_SOURCE_INDEX is not None:
    cfg["full_reference"] = FULL_SOURCE_INDEX
label = f"{PROFILE}_{SPLIT}_" + ("fullref" if cfg["full_reference"] else "subsetref")
RUN = RUN_ROOT / label
CONFIG = WORK / (label+".json")
CONFIG.write_text(json.dumps(cfg, indent=2))
def stage(name, *args):
    command([sys.executable,"-u","-m","denovogen.cli",name,"--config",CONFIG,
             "--run",RUN,"--device",DEVICE,*args])
print("Source:",source,"Run:",RUN)
print(json.dumps({k:cfg[k] for k in ("max_rows","epochs","updates","seeds","eval_samples","workloads")},indent=2))

## Prepare identities, split and token caches
The exact identity retains stereochemistry, disconnected fragments and tautomer form. Tokenization must round-trip. The vocabulary is fitted on train. In scaffold mode, inspect the acyclic group size because grouping all acyclic structures can unbalance the split. The same dataset directory is reused for every arm.

In [ ]:
stage("prepare","--source",source)
def read_json(path):
    return json.loads(path.read_text())
manifest = read_json(RUN/"data"/"manifest.json")
print(json.dumps({k:manifest[k] for k in ("source_raw_rows","selected_rows","split_counts","reference_identities",
      "complete_source_reference","changed_training_views","acyclic_group_size","counters")},indent=2))

## Train the two LSTM variants and select the common prior
Each epoch covers all training identities. The randomized variant alternates a seeded, equal-token-length molecular spelling with the canonical view. Validation loss chooses the epoch within each model. Independent validation generations choose the common prior. PAD is excluded from token loss and accuracy, while EOS is counted.

In [ ]:
stage("pretrain")
stage("select")
selection = read_json(RUN/"selection.json")
print("Selected prior:",selection["selected"])
for k,v in selection["variants"].items():
    print(k, "validation NLL",round(v["val_nll"],4),"validation validity",v["generation"]["validity"])

## Fine-tune six policies with REINFORCE
All arms start from the same checkpoint. R1 rewards validity, R2 adds training novelty, R3 adds QED to validity, R4 combines all three, R5 adds a verified SMARTS hit and R6 discounts repeated scaffolds. Sampled log probabilities include the actual termination action. A leave-one-out baseline reduces variance and a frozen-prior penalty limits drift. If a session expires, rerun this cell. To run one arm and seed, use e.g. `stage("rl","--arm","4","--seed","11")`.

In [ ]:
stage("rl")
print("Completed RL checkpoints:",len(list((RUN/"rl").glob("R*/seed_*/final.pt"))))

## Evaluate on a fixed budget
The frozen prior, both pretrained variants and six learned policies use common final evaluation seeds. Outputs include failed attempts, truncations, canonical IDs, exact novelty flags, QED, fragment matches and descriptors. Panel diagnostics add synthetic accessibility, PAINS alerts and nearest-neighbor similarity. A blank full-source novelty value means the required full reference was unavailable.

In [ ]:
stage("evaluate")
example = RUN/"final"/"R4"/f"seed_{cfg['seeds'][0]}"
print("Generated rows:",sorted(example.glob("chunk_*.csv"))[:2])
print({k:read_json(example/"metrics.json")[k] for k in
       ("attempts","valid","unique_valid","novelty_train","novelty_source","mean_qed","diversity")})

## Benchmark and inspect results
The scale benchmark picks one RL arm using validation. It runs both serial and batched decoding at the smallest configured workload and batched decoding at larger sizes. Workloads count attempted outputs, so failed molecules remain visible. Read end-to-end throughput and valid unique molecules per second together.

In [ ]:
stage("benchmark")
stage("report")
from IPython.display import display, Markdown, Image
display(Markdown((RUN/"report"/"RESULTS.md").read_text()))
for name in ["model_comparison.png","rl_diagnostics.png","first_valid_structures.png"]:
    path = RUN/"report"/name
    if path.exists():
        display(Image(filename=str(path)))

## Save the outputs
The compact results ZIP contains reports, plots, diagnostics, dataset inventory, per-seed summaries, and generated CSVs. Model weights, token caches and the SQLite identity database remain in RUN. For restoration on another machine, persist the entire run directory or enable EXPORT_RESUMABLE_ZIP. Choose a destination with sufficient disk space before archiving a large run.

In [ ]:
result_zip = RUN_ROOT / (label+"_results.zip")
with zipfile.ZipFile(result_zip,"w",zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN.rglob("*")):
        if not f.is_file():
            continue
        rel = f.relative_to(RUN)
        if f.name.startswith("chunk_") and f.suffix!=".csv":
            continue
        if f.suffix in {".json",".csv",".png",".md"}:
            z.write(f,rel)
print("Results ZIP:",result_zip)
print("Checkpoints and prepared data:",RUN)
if EXPORT_RESUMABLE_ZIP:
    print(shutil.make_archive(str(RUN_ROOT/(label+"_complete")), "zip", RUN.parent, RUN.name))